In [ ]:
# ---------------------------------------------------------------------------
# DATOS: acceso requiere infraestructura autorizada (Databricks).
# Los notebooks documentan el codigo analitico con fines de transparencia.
#
# Esquema esperado - gold_nps_features_groups (una fila por docente-respuesta):
#   nps_score              int     Puntuacion NPS (0-10)
#   nps_category           str     'detractor' | 'passive' | 'promoter'
#   n_sessions             int     Numero de sesiones en ventana 90 dias
#   clicks_per_session     float   Media de clics por sesion
#   n_distinct_features    int     Numero de funcionalidades distintas usadas
#   total_feature_clicks   int     Total de clics en la ventana
#   clicks_{group}         int     Clics totales por grupo funcional
#   rate_{group}           float   Clics del grupo / sesiones totales
#   intensity_{group}      float   Clics del grupo / sesiones con ese grupo
#   sessions_{group}       int     Sesiones con al menos 1 clic en ese grupo
#   (grupos: grading, modules, discussions, assignments, quizzes,
#    navigation, announcements, people, analytics, admin, otros)
# ---------------------------------------------------------------------------
raise NotImplementedError('Acceso a datos requiere infraestructura autorizada.')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import kruskal, spearmanr, mannwhitneyu
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi': 150,
    'savefig.bbox': 'tight'
})
PALETTE = {'detractor': '#d62728', 'passive': '#ff7f0e', 'promoter': '#2ca02c'}
ORDER = ['detractor', 'passive', 'promoter']
LABELS = {'detractor': 'Detractor', 'passive': 'Indiferente', 'promoter': 'Promotor'}
FIGURES_DIR = os.path.join(os.path.dirname(os.getcwd()), 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
print(f'Figuras en: {FIGURES_DIR}')


In [0]:
# Grupos funcionales y columnas derivadas
GROUPS = ['grading', 'modules', 'discussions', 'assignments', 'quizzes',
          'navigation', 'announcements', 'people', 'analytics', 'admin', 'otros']

GROUP_LABELS = {
    'grading':       'Evaluación y calificación',
    'modules':       'Módulos y contenido',
    'discussions':   'Debates y comunicación',
    'assignments':   'Tareas y entregas',
    'quizzes':       'Evaluaciones y tests',
    'navigation':    'Navegación',
    'announcements': 'Anuncios',
    'people':        'Gestión de estudiantes',
    'analytics':     'Analítica e informes',
    'admin':         'Administración',
    'otros':         'Otros',
}

clicks_cols    = [f'clicks_{g}'    for g in GROUPS]
rate_cols      = [f'rate_{g}'      for g in GROUPS]
intensity_cols = [f'intensity_{g}' for g in GROUPS]
sessions_cols  = [f'sessions_{g}'  for g in GROUPS]
print('Constantes definidas:', len(GROUPS), 'grupos funcionales')

## 1. Descripción de la población

In [0]:
# Tabla resumen del dataset
n_teachers    = len(df)
total_events  = df['total_feature_clicks'].sum()
avg_clicks    = df['total_feature_clicks'].mean()
median_clicks = df['total_feature_clicks'].median()
avg_sessions  = df['n_sessions'].mean()
avg_features  = df['n_distinct_features'].mean()

summary = pd.DataFrame({
    'Métrica': [
        'N docentes con NPS y eventos',
        'Total eventos en ventana 90d',
        'Media clics por docente',
        'Mediana clics por docente',
        'Media sesiones por docente',
        'Media grupos funcionales distintos/docente',
        'Ventana de observación',
        'Grupos funcionales instrumentados',
    ],
    'Valor': [
        f'{n_teachers:,}',
        f'{total_events:,.0f}',
        f'{avg_clicks:,.1f}',
        f'{median_clicks:,.1f}',
        f'{avg_sessions:.1f}',
        f'{avg_features:.1f}',
        '90 días previos a la respuesta NPS',
        '12',
    ]
})
print(summary.to_string(index=False))

In [0]:
# Niveles de actividad basados en total_feature_clicks
df['total_feature_clicks'] = df['total_feature_clicks'].astype(float)
q1  = df['total_feature_clicks'].quantile(0.25)
q3  = df['total_feature_clicks'].quantile(0.75)
p95 = df['total_feature_clicks'].quantile(0.95)

df['activity_level'] = pd.cut(
    df['total_feature_clicks'],
    bins=[-1, q1, q3, p95, float('inf')],
    labels=['Bajo (<Q1)', 'Medio (Q1–Q3)', 'Alto (Q3–p95)', 'Power user (>p95)']
)
level_dist = df['activity_level'].value_counts().reindex(
    ['Bajo (<Q1)', 'Medio (Q1–Q3)', 'Alto (Q3–p95)', 'Power user (>p95)'])
level_pct  = (level_dist / n_teachers * 100).round(1)
print(f'Umbrales — Q1: {q1:.0f} | Q3: {q3:.0f} | p95: {p95:.0f}')
print(pd.DataFrame({'n': level_dist, '%': level_pct}))

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma de clics totales (escala log)
axes[0].hist(df['total_feature_clicks'].clip(upper=df['total_feature_clicks'].quantile(0.99)),
             bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Clics totales en 90 días')
axes[0].set_ylabel('Número de docentes')
axes[0].set_title('Distribución de actividad total')
axes[0].axvline(median_clicks, color='orange', linestyle='--', label=f'Mediana: {median_clicks:.0f}')
axes[0].axvline(avg_clicks,    color='red',    linestyle=':',  label=f'Media: {avg_clicks:.0f}')
axes[0].legend()

# Box plot de clics por nivel de actividad
level_data = [df[df['activity_level'] == lv]['total_feature_clicks'].clip(upper=p95 * 1.5).values
              for lv in level_dist.index]
bx = axes[1].boxplot(level_data, labels=['Bajo', 'Medio', 'Alto', 'Power'], patch_artist=True)
colors_lv = ['#a8d8ea', '#a7c957', '#f4a261', '#e63946']
for patch, color in zip(bx['boxes'], colors_lv):
    patch.set_facecolor(color); patch.set_alpha(0.8)
axes[1].set_ylabel('Clics totales')
axes[1].set_title('Actividad por nivel de usuario')

plt.suptitle('Sección 1 — Descripción de la población', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}04_activity_histogram.png')
plt.show()
print('Fig. 04_activity_histogram.png guardada')

## 2. Distribución del NPS

In [0]:
# Tabla de frecuencias por puntuación
nps_dist = df['nps_score'].value_counts().sort_index()
nps_pct  = (nps_dist / len(df) * 100).round(1)
nps_table = pd.DataFrame({'n': nps_dist, '%': nps_pct})
print('Distribución de puntuaciones NPS (0–10):')
print(nps_table)

In [0]:
# Tabla por categoría + NPS neto
cat_dist = df['nps_category'].value_counts()
cat_pct  = (cat_dist / len(df) * 100).round(1)
cat_table = pd.DataFrame({'n': cat_dist, '%': cat_pct}).loc[ORDER]
print('Distribución por categoría NPS:')
print(cat_table)

nps_neto = cat_pct.get('promoter', 0) - cat_pct.get('detractor', 0)
print(f'\nNPS neto = %promotores − %detractores = {cat_pct.get("promoter",0):.1f}% − {cat_pct.get("detractor",0):.1f}% = {nps_neto:.1f}')

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histograma por puntuación
bar_colors = [PALETTE['detractor'] if s <= 6 else PALETTE['passive'] if s <= 8 else PALETTE['promoter']
              for s in range(11)]
axes[0].bar(range(11), [nps_dist.get(i, 0) for i in range(11)],
            color=bar_colors, edgecolor='white')
axes[0].set_xlabel('Puntuación NPS')
axes[0].set_ylabel('N docentes')
axes[0].set_title('Distribución NPS (0–10)')
axes[0].set_xticks(range(11))

# Barras por categoría
cat_vals   = [cat_dist.get(c, 0) for c in ORDER]
cat_colors = [PALETTE[c] for c in ORDER]
bars = axes[1].bar([LABELS[c] for c in ORDER], cat_vals, color=cat_colors, edgecolor='white')
for bar, v in zip(bars, cat_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
                 f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=9)
axes[1].set_ylabel('N docentes')
axes[1].set_title('Distribución por categoría')
axes[1].set_ylim(0, np.max(cat_vals) * 1.15)

# Barras con porcentaje — NPS neto
bar_sizes  = [cat_dist.get(c, 0) for c in ORDER]
bar_labels_nps = [LABELS[c] for c in ORDER]
bar_colors_nps = [PALETTE[c] for c in ORDER]
bars_d = axes[2].bar(bar_labels_nps, bar_sizes, color=bar_colors_nps, edgecolor='white')
for bar, v in zip(bars_d, bar_sizes):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=9)
axes[2].set_ylabel('N docentes')
axes[2].set_title(f'NPS neto: {nps_neto:+.1f}')
axes[2].set_ylim(0, np.max(bar_sizes) * 1.2)

plt.suptitle('Sección 2 — Distribución del NPS', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}01_nps_distribution.png')
plt.show()
print('Figura 01 guardada')

## 3. Análisis de uso de grupos funcionales

In [0]:
# Métricas por grupo: total clics, avg, mediana, penetración
group_stats = []
for g in GROUPS:
    col = f'clicks_{g}'
    n_users_with_use = (df[col] > 0).sum()
    group_stats.append({
        'grupo':         GROUP_LABELS[g],
        'total_clicks':  int(df[col].sum()),
        'avg_clicks':    round(df[col].mean(), 1),
        'median_clicks': round(df[col].median(), 1),
        'penetracion_%': round(n_users_with_use / len(df) * 100, 1),
    })

group_df = pd.DataFrame(group_stats).sort_values('total_clicks', ascending=False)
group_df['% acum clicks'] = (group_df['total_clicks'].cumsum() /
                               group_df['total_clicks'].sum() * 100).round(1)
print('Métricas por grupo funcional:')
print(group_df.to_string(index=False))

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 12 por total_clicks
gdf_sorted = group_df.sort_values('total_clicks')
axes[0].barh(gdf_sorted['grupo'], gdf_sorted['total_clicks'] / 1e6,
             color='steelblue', edgecolor='white')
axes[0].set_xlabel('Millones de clics')
axes[0].set_title('Total de clics por grupo funcional')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))

# Top 12 por penetración
gdf_pen = group_df.sort_values('penetracion_%')
colors_pen = ['#2ca02c' if p >= 50 else '#ff7f0e' if p >= 20 else '#d62728'
              for p in gdf_pen['penetracion_%']]
axes[1].barh(gdf_pen['grupo'], gdf_pen['penetracion_%'], color=colors_pen, edgecolor='white')
axes[1].set_xlabel('% de docentes con al menos 1 clic')
axes[1].set_title('Penetración por grupo funcional')
axes[1].axvline(50, color='gray', linestyle='--', alpha=0.5, label='50%')
axes[1].legend()

plt.suptitle('Sección 3 — Rankings de uso', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}05_feature_rankings.png')
plt.show()

In [0]:
# Pareto chart: % acumulado de clics por grupo funcional
fig, ax1 = plt.subplots(figsize=(12, 5))
gdf_par = group_df.sort_values('total_clicks', ascending=False).reset_index(drop=True)

ax1.bar(range(len(gdf_par)), gdf_par['total_clicks'] / 1e6, color='steelblue', edgecolor='white')
ax1.set_xticks(range(len(gdf_par)))
ax1.set_xticklabels([g[:20] for g in gdf_par['grupo']], rotation=40, ha='right', fontsize=12)
ax1.set_ylabel('Millones de clics', fontsize=13)
ax1.set_title('Diagrama de Pareto — Clics por grupo funcional', fontsize=15)
ax1.tick_params(axis='x', labelsize=12)
ax1.tick_params(axis='y', labelsize=12)

ax2 = ax1.twinx()
ax2.plot(range(len(gdf_par)), gdf_par['% acum clicks'], 'o--', color='orange',
         linewidth=2, markersize=7, label='% acumulado')
ax2.axhline(80, color='red', linestyle=':', alpha=0.6, label='80%')
ax2.set_ylabel('% acumulado', fontsize=13)
ax2.set_ylim(0, 105)
ax2.tick_params(axis='y', labelsize=12)
ax2.legend(loc='center right', fontsize=12)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}07_pareto_groups.png', dpi=300)
plt.show()
print('Fig. 07_pareto_groups.png guardada')

In [0]:
# Heatmap de intensidad (intensity_*) por grupo × categoría NPS
intensity_means = df.groupby('nps_category')[intensity_cols].mean()[
    [f'intensity_{g}' for g in GROUPS]
]
intensity_means.columns = [GROUP_LABELS[g] for g in GROUPS]
intensity_means = intensity_means.loc[ORDER]
intensity_means.index = [LABELS[c] for c in ORDER]

fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(intensity_means, cmap='YlOrRd', annot=True, fmt='.1f',
            annot_kws={'size': 11}, ax=ax,
            cbar_kws={'label': 'Clics/sesión con ese grupo'})
ax.set_title('Intensidad media por grupo funcional y categoría NPS')
ax.set_xlabel('Grupo funcional')
ax.set_ylabel('Categoría NPS')
ax.set_yticklabels([l.get_text() for l in ax.get_yticklabels()], rotation=0, ha='right', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=11)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}08_intensity_heatmap.png')
plt.show()
print('Fig. 08_intensity_heatmap.png guardada')

## 4. Uso de grupos funcionales por categoría NPS

In [0]:
# Estadísticos descriptivos por grupo (rate_*) y test Kruskal-Wallis
results = []
for g in GROUPS:
    col = f'rate_{g}'
    groups_data = [df.loc[df.nps_category == cat, col].dropna() for cat in ORDER]
    try:
        stat, pval = kruskal(*groups_data)
    except ValueError:
        stat, pval = 0.0, float('nan')
    results.append({
        'grupo':           GROUP_LABELS[g],
        'mean_detractor':  round(groups_data[0].mean(), 4),
        'mean_passive':    round(groups_data[1].mean(), 4),
        'mean_promoter':   round(groups_data[2].mean(), 4),
        'kruskal_H':       round(stat, 2),
        'p_value':         round(pval, 4) if not np.isnan(pval) else np.nan,
        'sig':             '***' if (not np.isnan(pval) and pval < 0.001) else
                           '**'  if (not np.isnan(pval) and pval < 0.01)  else
                           '*'   if (not np.isnan(pval) and pval < 0.05)  else ''
    })

kruskal_df = pd.DataFrame(results).sort_values('kruskal_H', ascending=False)
print('Test de Kruskal-Wallis por grupo funcional (rate_*):')
print(kruskal_df.to_string(index=False))
kruskal_df.to_csv(f'{FIGURES_DIR}tabla_kruskal_grupos.csv', index=False)

In [0]:
# Gráfico de barras agrupadas: medias de rate_* por categoría NPS
fig, ax = plt.subplots(figsize=(7, 3.2))  # tamaño reducido para docx

x         = np.arange(len(GROUPS))
width     = 0.28
group_lbls = [GROUP_LABELS[g][:15] for g in GROUPS]

for i, cat in enumerate(ORDER):
    means = [df.loc[df.nps_category == cat, f'rate_{g}'].mean() for g in GROUPS]
    bars  = ax.bar(x + (i - 1) * width, means, width, label=LABELS[cat],
                   color=PALETTE[cat], alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(group_lbls, rotation=40, ha='right', fontsize=10)  # aumentar fontsize
ax.set_ylabel('Clics por sesión (tasa media)', fontsize=11)
ax.set_title('Tasa de uso por grupo funcional y categoría del índice neto de promotores', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}09_grouped_bars_rate.png', dpi=300)  # alta resolución
plt.show()
print('Fig. 09_grouped_bars_rate.png guardada')

In [0]:
# Gráfico de diferencias: rate_promoter − rate_detractor por grupo
diffs = []
for g in GROUPS:
    col    = f'rate_{g}'
    m_pro  = df.loc[df.nps_category == 'promoter',  col].mean()
    m_det  = df.loc[df.nps_category == 'detractor', col].mean()
    diffs.append({'grupo': GROUP_LABELS[g], 'diff': m_pro - m_det})

diff_df = pd.DataFrame(diffs).sort_values('diff')
colors_diff = ['#2ca02c' if d > 0 else '#d62728' for d in diff_df['diff']]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(diff_df['grupo'], diff_df['diff'], color=colors_diff, edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Diferencia de tasa media (Promotor − Detractor)')
ax.set_title('Grupos donde los promotores usan más/menos que los detractores')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}10_difference_chart.png')
plt.show()
print('Fig. 10_difference_chart.png guardada')

In [0]:
# Box plots de los 6 grupos más discriminantes (mayor H de Kruskal-Wallis)
top6_groups = kruskal_df.head(6)['grupo'].tolist()
top6_keys   = [g for g in GROUPS if GROUP_LABELS[g] in top6_groups]

fig, axes = plt.subplots(2, 3, figsize=(9, 5))  # tamaño reducido para docx
axes = axes.flatten()

for i, g in enumerate(top6_keys):
    col  = f'rate_{g}'
    p99  = df[col].quantile(0.99)
    data = [df.loc[df.nps_category == cat, col].fillna(0).clip(upper=p99).values for cat in ORDER]
    bp   = axes[i].boxplot(data, labels=[LABELS[c] for c in ORDER], patch_artist=True)
    for patch, cat in zip(bp['boxes'], ORDER):
        patch.set_facecolor(PALETTE[cat]); patch.set_alpha(0.7)
    axes[i].set_title(GROUP_LABELS[g], fontsize=11)
    axes[i].set_ylabel('Clics/sesión (p99 win.)', fontsize=10)
    axes[i].tick_params(axis='x', labelsize=10)
    axes[i].tick_params(axis='y', labelsize=9)

plt.suptitle('Distribución de uso por categoría NPS — Top 6 grupos', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}02_boxplots_features_by_nps.png', dpi=300)
plt.show()
print('Fig. 02 guardada')

## 5. Intensidad del comportamiento

In [0]:
# clicks_per_session por categoría NPS: violin + Kruskal-Wallis
col = 'clicks_per_session'
p99 = df[col].fillna(0).quantile(0.99)
cap = p99 if p99 > 0 else df[col].fillna(0).max()
data_groups = [df.loc[df.nps_category == cat, col].fillna(0).clip(upper=cap).values for cat in ORDER]

stat, pval = kruskal(*data_groups)
print(f'Kruskal-Wallis clicks_per_session — H={stat:.2f}, p={pval:.4e}')
print(df.groupby('nps_category')[col].agg(['mean', 'median', 'std']).loc[ORDER].round(2))

fig, ax = plt.subplots(figsize=(7, 5))
# Violin requiere varianza en los datos; si algún grupo es constante, usar boxplot como fallback
_has_variance = all(arr.std() > 0 for arr in data_groups if len(arr) > 0)
if _has_variance:
    parts = ax.violinplot(data_groups, positions=[1, 2, 3], showmedians=True)
    for pc, cat in zip(parts['bodies'], ORDER):
        pc.set_facecolor(PALETTE[cat]); pc.set_alpha(0.7)
else:
    bp = ax.boxplot(data_groups, labels=[LABELS[c] for c in ORDER], patch_artist=True)
    for patch, cat in zip(bp['boxes'], ORDER):
        patch.set_facecolor(PALETTE[cat]); patch.set_alpha(0.7)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels([LABELS[c] for c in ORDER])
ax.set_ylabel('Clics por sesión (p99 win.)')
ax.set_title(f'Intensidad de uso por sesión\nKruskal-Wallis H={stat:.1f}, p={pval:.3e}')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}11_violin_clicks_per_session.png')
plt.show()
print('Fig. 11 guardada')

In [0]:
# Sesiones por semana: n_sessions / (90/7)
df['sessions_per_week'] = df['n_sessions'].astype(float) / (90 / 7)
col  = 'sessions_per_week'
p99  = df[col].quantile(0.99)
data_groups = [df.loc[df.nps_category == cat, col].clip(upper=p99).values for cat in ORDER]

stat, pval = kruskal(*data_groups)
print(f'Kruskal-Wallis sesiones/semana — H={stat:.2f}, p={pval:.4e}')
print(df.groupby('nps_category')[col].agg(['mean', 'median']).loc[ORDER].round(2))

fig, ax = plt.subplots(figsize=(7, 5))
parts = ax.violinplot(data_groups, positions=[1, 2, 3], showmedians=True)
for pc, cat in zip(parts['bodies'], ORDER):
    pc.set_facecolor(PALETTE[cat]); pc.set_alpha(0.7)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels([LABELS[c] for c in ORDER])
ax.set_ylabel('Sesiones por semana')
ax.set_title(f'Frecuencia de sesiones\nKruskal-Wallis H={stat:.1f}, p={pval:.3e}')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}11b_violin_sessions_per_week.png')
plt.show()

In [0]:
# Duración de sesión (minutos) desde df_seq
df_seq['duration_min'] = (df_seq['session_end'] - df_seq['session_start']).dt.total_seconds() / 60
p99_dur = df_seq['duration_min'].quantile(0.99)

dur_groups = [
    df_seq.loc[df_seq.nps_category == cat, 'duration_min'].clip(upper=p99_dur).values
    for cat in ORDER
]
stat, pval = kruskal(*dur_groups)
print(f'Kruskal-Wallis duración sesión (min) — H={stat:.2f}, p={pval:.4e}')
print(df_seq.groupby('nps_category')['duration_min'].agg(['mean', 'median', 'std']).loc[ORDER].round(2))

fig, ax = plt.subplots(figsize=(7, 5))
parts = ax.violinplot(dur_groups, positions=[1, 2, 3], showmedians=True)
for pc, cat in zip(parts['bodies'], ORDER):
    pc.set_facecolor(PALETTE[cat]); pc.set_alpha(0.7)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels([LABELS[c] for c in ORDER])
ax.set_ylabel('Duración (min, p99 win.)')
ax.set_title(f'Duración de sesión por categoría NPS\nKruskal-Wallis H={stat:.1f}, p={pval:.3e}')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}11c_violin_session_duration.png')
plt.show()

## 6. Diversidad de grupos funcionales

In [0]:
# n_distinct_features (grupos distintos con >0 clics por docente)
df['n_active_groups'] = (df[clicks_cols] > 0).sum(axis=1)

for col, label in [('n_distinct_features', 'Funcionalidades distintas'),
                   ('n_active_groups',     'Grupos activos (de 11)')]:
    data_groups = [df.loc[df.nps_category == cat, col].values for cat in ORDER]
    stat, pval  = kruskal(*data_groups)
    print(f'\n{label} — Kruskal-Wallis H={stat:.2f}, p={pval:.4e}')
    print(df.groupby('nps_category')[col].agg(['mean', 'median']).loc[ORDER].round(2))

fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))  # tamaño reducido para docx
for ax, col, label in zip(axes,
                           ['n_distinct_features', 'n_active_groups'],
                           ['Funcionalidades individuales distintas', 'Grupos funcionales activos']):
    p99  = df[col].astype(float).quantile(0.99)
    data = [df.loc[df.nps_category == cat, col].clip(upper=p99).astype(float).values for cat in ORDER]
    parts = ax.violinplot(data, positions=[1, 2, 3], showmedians=True)
    for pc, cat in zip(parts['bodies'], ORDER):
        pc.set_facecolor(PALETTE[cat]); pc.set_alpha(0.7)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels([LABELS[c] for c in ORDER], fontsize=10)  # etiquetas más grandes
    ax.set_ylabel(label, fontsize=10)  # etiquetas más grandes
    ax.set_title(f'Diversidad — {label}', fontsize=10)  # etiquetas más grandes

plt.suptitle('Diversidad de uso', fontsize=10)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}12_violin_diversity.png')
plt.show()
print('Fig. 12 guardada')

## 7. Comportamiento temporal

In [0]:
# Uso por hora del día y día de la semana (desde df_seq)
df_seq['hour']    = df_seq['session_start'].dt.hour
df_seq['weekday'] = df_seq['session_start'].dt.dayofweek  # 0=lunes

# Heatmap hora × NPS
hour_nps = df_seq.groupby(['nps_category', 'hour']).size().unstack(fill_value=0)
hour_nps = hour_nps.loc[ORDER] if all(c in hour_nps.index for c in ORDER) else hour_nps
hour_nps_norm = hour_nps.div(hour_nps.sum(axis=1), axis=0) * 100  # % dentro de cada categoría

fig, axes = plt.subplots(2, 1, figsize=(9, 5))  # tamaño reducido para docx

sns.heatmap(hour_nps_norm, cmap='Blues', annot=False, ax=axes[0],
            cbar_kws={'label': '% de sesiones'})
axes[0].set_title('Distribución horaria de sesiones por categoría del índice neto de promotores', fontsize=11)
axes[0].set_xlabel('Hora del día', fontsize=10)
axes[0].set_ylabel('Categoría NPS', fontsize=10)
axes[0].set_yticklabels([LABELS[c] for c in ORDER if c in hour_nps.index], rotation=0, fontsize=10)
axes[0].set_xticklabels([str(h) for h in hour_nps_norm.columns], fontsize=9)

# Líneas por hora
hours = range(24)
for cat in ORDER:
    if cat in hour_nps_norm.index:
        axes[1].plot(hours, hour_nps_norm.loc[cat],
                     label=LABELS[cat], color=PALETTE[cat], linewidth=2)
axes[1].set_xlabel('Hora del día', fontsize=10)
axes[1].set_ylabel('% de sesiones', fontsize=10)
axes[1].set_title('Perfil de actividad horaria', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].set_xticks(range(0, 24, 2))
axes[1].set_xticklabels([str(h) for h in range(0, 24, 2)], fontsize=9)
axes[1].tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}13_temporal_heatmap.png', dpi=300)  # alta resolución
plt.show()
print('Fig. 13 guardada')

In [0]:
# Uso por día de la semana
day_labels = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
weekday_nps = df_seq.groupby(['nps_category', 'weekday']).size().unstack(fill_value=0)
weekday_norm = weekday_nps.div(weekday_nps.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(7)
width = 0.28
for i, cat in enumerate(ORDER):
    if cat in weekday_norm.index:
        vals = [weekday_norm.loc[cat, d] if d in weekday_norm.columns else 0 for d in range(7)]
        ax.bar(x + (i - 1) * width, vals, width, label=LABELS[cat],
               color=PALETTE[cat], alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(day_labels)
ax.set_ylabel('% de sesiones')
ax.set_title('Distribución de sesiones por día de la semana')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}13b_weekday_distribution.png')
plt.show()

## 8. Co-ocurrencia de grupos funcionales

In [0]:
# Matriz de co-ocurrencia: % de docentes que usan ambos grupos (sessions_* > 0)
n = len(df)
cooc = pd.DataFrame(index=GROUPS, columns=GROUPS, dtype=float)

for g1 in GROUPS:
    for g2 in GROUPS:
        if g1 == g2:
            cooc.loc[g1, g2] = (df[f'sessions_{g1}'] > 0).mean() * 100
        else:
            both = ((df[f'sessions_{g1}'] > 0) & (df[f'sessions_{g2}'] > 0)).sum()
            cooc.loc[g1, g2] = both / n * 100

cooc.index   = [GROUP_LABELS[g][:18] for g in GROUPS]
cooc.columns = [GROUP_LABELS[g][:18] for g in GROUPS]

fig, ax = plt.subplots(figsize=(8, 6))  # tamaño reducido para docx
mask = np.triu(np.ones_like(cooc, dtype=bool), k=1)
sns.heatmap(cooc.astype(float), mask=mask, cmap='YlGnBu', vmin=0, vmax=100,
            annot=True, fmt='.0f', annot_kws={'size': 7}, ax=ax,
            cbar_kws={'label': '% de docentes'})
ax.set_title('Co-ocurrencia de grupos funcionales (% docentes que usan ambos)', fontsize=11)
ax.set_xticklabels(cooc.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(cooc.index, rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}14_cooccurrence_matrix.png', dpi=300)
plt.show()
print('Fig. 14 guardada')

In [0]:
# Correlación de Spearman entre rate_* (reemplaza heatmap anterior)
rate_matrix = df[rate_cols].fillna(0).corr(method='spearman')
rate_matrix.index   = [GROUP_LABELS[g][:18] for g in GROUPS]
rate_matrix.columns = [GROUP_LABELS[g][:18] for g in GROUPS]

fig, ax = plt.subplots(figsize=(7, 5))  # tamaño reducido para docx
mask = np.triu(np.ones_like(rate_matrix, dtype=bool))
sns.heatmap(rate_matrix, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 8}, ax=ax,
            cbar_kws={'label': 'Correlación de Spearman'})
ax.set_title('Correlación de Spearman entre tasas de uso por grupo funcional', fontsize=11)
ax.set_xticklabels(rate_matrix.columns, rotation=40, ha='right', fontsize=9)
ax.set_yticklabels(rate_matrix.index, fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}03_correlation_matrix.png', dpi=300)
plt.show()
print('Fig. 03 guardada')

In [0]:
# Correlación de Spearman de cada rate_* con nps_score
spearman_results = []
for g in GROUPS:
    col = f'rate_{g}'
    rho, pval = spearmanr(df[col].fillna(0), df['nps_score'])
    spearman_results.append({
        'grupo': GROUP_LABELS[g],
        'rho':   round(rho, 3),
        'p_val': round(pval, 4),
        'sig':   '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    })

spearman_df = pd.DataFrame(spearman_results).sort_values('rho', ascending=False)
print('Correlaciones de Spearman con NPS (rate_*):')
print(spearman_df.to_string(index=False))
spearman_df.to_csv(f'{FIGURES_DIR}tabla_spearman_grupos.csv', index=False)

## 9. Calidad de datos y outliers

In [0]:
# Valores nulos por columna
null_pct = (df.isnull().sum() / len(df) * 100).round(1)
print('% valores nulos por columna (solo columnas con nulos):')
null_tab = null_pct[null_pct > 0]
print(null_tab if len(null_tab) > 0 else 'Sin valores nulos')

# Funcionalidades con uso disperso: % de docentes con 0 clics por grupo
print('\n% de docentes con 0 clics por grupo funcional:')
zero_pct = pd.DataFrame({
    'grupo': [GROUP_LABELS[g] for g in GROUPS],
    '% con 0 clics': [(df[f'clicks_{g}'] == 0).mean() * 100 for g in GROUPS]
}).sort_values('% con 0 clics', ascending=False)
print(zero_pct.to_string(index=False))

In [0]:
# Outliers extremos en total_feature_clicks
q95 = df['total_feature_clicks'].quantile(0.95)
q99 = df['total_feature_clicks'].quantile(0.99)
n_outliers = (df['total_feature_clicks'] > q99).sum()
print(f'total_feature_clicks — p95: {q95:,.0f} | p99: {q99:,.0f} | máx: {df["total_feature_clicks"].max():,.0f}')
print(f'Outliers (>p99): {n_outliers:,} docentes ({n_outliers/len(df)*100:.1f}%)')
print('\nDistribución por NPS de outliers (>p99):')
print(df[df['total_feature_clicks'] > q99]['nps_category'].value_counts())

# Perfil power users vs. resto
df['is_power'] = df['total_feature_clicks'] > q99
power_profile  = df.groupby('is_power')[['total_feature_clicks', 'n_sessions',
                                          'n_distinct_features', 'clicks_per_session']].mean().round(1)
power_profile.index = ['Regular', 'Power user (>p99)']
print('\nPerfil medio:')
print(power_profile)

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))  # tamaño reducido para docx
# Distribución NPS en power users vs resto
for i, (label, mask) in enumerate([('Uso Normal', ~df['is_power']), ('Uso Intenso (>p99)', df['is_power'])]):
    nps_cat = df[mask]['nps_category'].value_counts()
    wedges, texts, autotexts = axes[i].pie(
        [nps_cat.get(c, 0) for c in ORDER],
        labels=[LABELS[c] for c in ORDER],
        colors=[PALETTE[c] for c in ORDER],
        autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12}
    )
    for t in texts:
        t.set_fontsize(12)
    for at in autotexts:
        at.set_fontsize(12)
    axes[i].set_title(f'{label} (n={mask.sum():,})', fontsize=13)

plt.suptitle('Distribución NPS: Usarios de uso intenso vs. usuarios uso normal', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}15_outliers_profile.png', dpi=300)
plt.show()
print('Fig. 15 guardada')

In [0]:
# Guardar tablas resumen para el TFM
group_df.to_csv(f'{FIGURES_DIR}tabla_grupos_metricas.csv', index=False)
diff_df.to_csv(f'{FIGURES_DIR}tabla_diferencias_rate.csv', index=False)
zero_pct.to_csv(f'{FIGURES_DIR}tabla_zeros_por_grupo.csv', index=False)
power_profile.to_csv(f'{FIGURES_DIR}tabla_power_vs_regular.csv')

saved = [
    '01_nps_distribution.png', '02_boxplots_features_by_nps.png', '03_correlation_matrix.png',
    '04_activity_histogram.png', '05_feature_rankings.png', '06_nps_donut.png',
    '07_pareto_groups.png', '08_intensity_heatmap.png', '09_grouped_bars_rate.png',
    '10_difference_chart.png', '11_violin_clicks_per_session.png', '11b_violin_sessions_per_week.png',
    '11c_violin_session_duration.png', '12_violin_diversity.png', '13_temporal_heatmap.png',
    '13b_weekday_distribution.png', '14_cooccurrence_matrix.png', '15_outliers_profile.png',
    'tabla_kruskal_grupos.csv', 'tabla_spearman_grupos.csv', 'tabla_grupos_metricas.csv',
    'tabla_diferencias_rate.csv', 'tabla_zeros_por_grupo.csv', 'tabla_power_vs_regular.csv',
]

print(f'EDA completado. {len(saved)} ficheros guardados en: {FIGURES_DIR}')
for f in saved:
    print(f'  {f}')